# GNN for edge classification to link the tracksters

Here, we will create an Edge classification graph that will link the tracksters belonging to the same particle. We will have the following properties:
1. Nodes: Tracksters
2. Node features: Trackster features
3. True label: From Associations tree
4. Edge features: .....

## Opening the root file

In [ ]:
import uproot
import awkward as ak
import numpy as np

file=uproot.open("flat_tree_for_Neutron.root")

tracksters=file["ticlDumper/CLUE3DHighTree"].arrays(library="ak")
simtracksters=file["ticlDumper/SimTrackstersTree"].arrays(library="ak")
associations=file["ticlDumper/associations"].arrays(library="ak")

In [ ]:
tracksters

In [ ]:
associations

In [ ]:
associations[0]

In [ ]:
associations[0]["CLUE3DHigh_recoToSim_CP"]

In [ ]:
associations[0]["CLUE3DHigh_recoToSim_CP_score"]

In [ ]:
associations[2]["CLUE3DHigh_recoToSim_CP_score"]

In [ ]:
associations[0]["CLUE3DHigh_simToReco_CP_score"]

In [ ]:
test_event=tracksters[0]

In [ ]:
test_event

In [ ]:
test_event.time

## Extracting the graph features

### Getting the node features

The tracksters are the nodes and the node features are the characteristics of the tracksters

In [ ]:
def get_node_features(event):
    feats=np.stack([
        event["time"],
        event["raw_energy"],
        event["raw_em_energy"],
        event["raw_pt"],
        event["barycenter_x"],
        event["barycenter_y"],
        event["barycenter_z"],
        event["barycenter_eta"],
        event["barycenter_phi"],
        event["EV1"],
        event["EV2"],
        event["EV3"],
        event["sigmaPCA1"],
        event["sigmaPCA2"],
        event["sigmaPCA3"]
    ],axis=1)
    
    return np.array(feats)

In [ ]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
node_features.shape

### Building the graph edges

In [ ]:
from sklearn.neighbors import NearestNeighbors

def build_edges(event,k=8):
    x=np.array(event["barycenter_x"])
    y=np.array(event["barycenter_y"])
    z=np.array(event["barycenter_z"])
    coords=np.stack([x,y,z],axis=1)
    # Fit KNN-determine the KNN
    n_nodes=len(coords)
    k_eff=min(k,n_nodes-1)
    if n_nodes<2:
        return np.empty((2,0),dtype=int)
    knn=NearestNeighbors(n_neighbors=k_eff+1,algorithm='auto').fit(coords)
    dist,idx=knn.kneighbors(coords)
    #Creating edges
    edges=[]
    for i,nbrs in enumerate(idx):
        for j in nbrs[1:]:
            edges.append([i,j])
    edges=np.array(edges).T
    return edges

In [ ]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
edges=build_edges(test_event)
#edges
edges.shape

### Build Edge Features

In [ ]:
def build_edge_features(event,edge_index):
    x=np.array(event["barycenter_x"])
    y=np.array(event["barycenter_y"])
    z=np.array(event["barycenter_z"])
    eta=np.array(event["barycenter_eta"])
    phi=np.array(event["barycenter_phi"])
    t=np.array(event["time"])
    E=np.array(event["raw_energy"])
    
    features=[]
    
    for i,j in edge_index.T:
        dx=x[i]-x[j]
        dy=y[i]-y[j]
        dz=z[i]-z[j]
        dist=np.sqrt(dx*dx+dy*dy+dz*dz)
        deta=eta[i]-eta[j]
        dphi=phi[i]-phi[j]
        dt=t[i]-t[j]
        dE=E[i]-E[j]
        features.append([dist,dz,deta,dphi,dt,dE])
    return np.array(features)

In [ ]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
edges=build_edges(test_event)
#edges
edge_features=build_edge_features(test_event,edges)
edge_features.shape

### Getting the truth mappings

In [ ]:
def get_truth_mappings(event_id):
    reco_to_sim=associations[event_id]["CLUE3DHigh_recoToSim_CP"]
    truth={}
    for i,match in enumerate(reco_to_sim):
        if len(match)>0:
            truth[i]=int(match[0])
    return truth

In [ ]:
#Checking the scores to get the proper threshold
all_scores = []

for ev in range(len(associations)):
    scores = associations[ev]["CLUE3DHigh_recoToSim_CP_score"]
    
    for s in scores:
        if len(s) > 0:
            all_scores.extend(s)

import numpy as np

all_scores = np.array(all_scores)

print("Min:", all_scores.min())
print("Max:", all_scores.max())
print("Mean:", all_scores.mean())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(all_scores, bins=50)
plt.xlabel("Reco-to-Sim Score")
plt.ylabel("Counts")
plt.yscale("log")
plt.show()

In [ ]:
print("50%:", np.percentile(all_scores, 50))
print("75%:", np.percentile(all_scores, 75))
print("90%:", np.percentile(all_scores, 90))
print("95%:", np.percentile(all_scores, 95))

In [ ]:
def get_truth_mappings(event_id,score_threshold=1e-4):
    reco_to_sim=associations[event_id]["CLUE3DHigh_recoToSim_CP"]
    reco_to_sim_score=associations[event_id]["CLUE3DHigh_recoToSim_CP_score"]
    truth={}
    for i,(matches,scores) in enumerate(zip(reco_to_sim,reco_to_sim_score)):
        if len(matches)==0:
            continue
        #Convert to numpy for safety
        scores=np.array(scores)
        #Finding the best match
        best_idx=np.argmax(scores)
        best_score=scores[best_idx]
        if best_score>score_threshold:
            truth[i]=int(matches[best_idx])
        else:
            truth[i]=-1 #Marked as noise
    return truth

In [ ]:
#Testing
get_truth_mappings(0)

### Building edge labels

In [ ]:
def build_edge_labels(edge_index,truth):
    labels=[]
    for i,j in edge_index.T:
        if i in truth and j in truth and truth[i]==truth[j]:
            labels.append(1)
        else:
            labels.append(0)
    return np.array(labels)

In [ ]:
def build_edge_labels(edge_index,truth):
    labels=[]
    for i,j in edge_index.T:
        ti=truth.get(int(i),-1)
        tj=truth.get(int(j),-1)
        if ti!=-1 and tj!=-1 and ti==tj:
            labels.append(1)
        else:
            labels.append(0)
    return np.array(labels)

In [ ]:
# Testing
event_id=0
test_event=tracksters[event_id]
node_features=get_node_features(test_event)
edge_index=build_edges(test_event)
#edges
edge_features=build_edge_features(test_event,edge_index)
#truth
truth=get_truth_mappings(event_id)
edge_labels=build_edge_labels(edge_index,truth)
edge_labels

## Building the graph object

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data,Dataset
from torch_geometric.loader import DataLoader

In [ ]:
def build_graph(event_id):
    event=tracksters[event_id]
    node_features=get_node_features(event)
    edge_index=build_edges(event)
    edge_attr=build_edge_features(event,edge_index)
    truth=get_truth_mappings(event_id)
    labels=build_edge_labels(edge_index,truth)
    data=Data(x=torch.tensor(node_features),
              edge_index=torch.tensor(edge_index),
              edge_attr=torch.tensor(edge_attr),
              y=torch.tensor(labels))
    return data

In [ ]:
def build_graph(event_id):
    event=tracksters[event_id]
    node_features=get_node_features(event)
    edge_index=build_edges(event)
    edge_attr=build_edge_features(event,edge_index)
    truth=get_truth_mappings(event_id)
    labels=build_edge_labels(edge_index,truth)
    data=Data(x=torch.tensor(node_features,dtype=torch.float),
              edge_index=torch.tensor(edge_index,dtype=torch.long),
              edge_attr=torch.tensor(edge_attr,dtype=torch.float)
             )
    data.edge_label=torch.tensor(labels,dtype=torch.float)
    return data

In [ ]:
#Testing- for the first event
event_idx=0
test_graph=build_graph(event_id=event_idx)
test_graph

## Building the dataset

In [ ]:
%%time
graph=[]
for i in range(1000):
    graph.append(build_graph(event_id=i))

In [ ]:
len(graph)

In [ ]:
graph[0]

## Splitting into training,Validation and Testing data

In [ ]:
from sklearn.model_selection import train_test_split
#First split: Train vs Temp(Val+Test)
train_data, temp_data= train_test_split(graph, test_size=0.3, random_state=42)

#Second split: Test vs Val
val_data, test_data= train_test_split(temp_data, test_size=0.7, random_state=42)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

## Using the dataloader

In [ ]:
from torch_geometric.loader import DataLoader
batch_size=32

train_loader=DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader=DataLoader(val_data, batch_size=batch_size, shuffle=True)
test_loader=DataLoader(test_data, batch_size=batch_size, shuffle=True)

## Implementing the GNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Edge_Classifier(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        super().__init__()
        #First GCN Layer
        self.conv1=GCNConv(input_dim,hidden_dim)
        #Stacking 6 Repeated blocks
        self.convs=nn.ModuleList()
        for i in range(6):
            self.convs.append(GCNConv(hidden_dim,hidden_dim))
        #Final layer
        self.final_conv=GCNConv(hidden_dim,hidden_dim)
        #Regularization
        self.dropout=nn.Dropout(p=0.1)
        #The Edge Classification MLP
        self.edge_mlp=nn.Sequential(
            nn.Linear(2*hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,1)
        )
    def forward(self,data):
        x,edge_index,batch=data.x,data.edge_index,data.batch
        #-----------Passing the GCN forward passes--------------#
        ##### First GCN ######
        x=self.conv1(x,edge_index)
        x=F.relu(x)
        x=self.dropout(x)
        ##### Six repeated convolutions #####
        for conv in self.convs:
            x=conv(x,edge_index)
            x=F.relu(x)
            x=self.dropout(x)
        ##### Final convolution #####
        x=self.final_conv(x,edge_index)
        node_embedding=x
        #------------ Edge Classification MLP -----------------#
        row,col=edge_index
        edge_features=torch.cat([x[row],x[col]],dim=1)
        edge_logits=self.edge_mlp(edge_features).view(-1)
        return edge_logits

In [ ]:
# Device setup
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ",device)

In [ ]:
node_dim=graph[0].x.shape[1]
model=Edge_Classifier(input_dim=node_dim,hidden_dim=64).to(device)

In [ ]:
model

## Training, Testing and Validation

In [ ]:
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
criterion=nn.BCEWithLogitsLoss()

### Training

In [ ]:
# Traiing loop
def train():
    model.train()
    total_loss=0.0
    for batch in train_loader:
        batch=batch.to(device)
        optimizer.zero_grad()
        #Forward pass
        out=model(batch)
        #Compute Loss
        loss=criterion(out,batch.edge_label)
        #Backpropagation
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(train_loader)

In [ ]:
# Validation Loop
def validate():
    model.eval()
    total_loss=0.0
    with torch.no_grad():
        for batch in val_loader:
            batch=batch.to(device)
            out=model(batch)
            loss=criterion(out,batch.edge_label)
            total_loss+=loss.item()
    return total_loss/len(val_loader)

In [ ]:
%%time
epochs=10
train_losses=[]
val_losses=[]
for epoch in range(1,epochs+1):
    train_loss=train()
    val_loss=validate()
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch}, Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

In [ ]:
#Visualizing the model performance
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
plt.plot(range(1,len(train_losses)+1),train_losses,label="Training loss",color="Blue")
plt.plot(range(1,len(val_losses)+1),val_losses,label="Validation loss",color="Red")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation loss vs Epochs")
plt.legend()
plt.grid()
plt.show()